In [3]:
# generar_cache.py — ejecutar localment
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import os
import random

CSV_PATH = "C:/Users/47173276T/Downloads/new_collect/collection.csv"
CACHE_PATH = "C:/Users/47173276T/Downloads/new_collect/bgg_mechanics_cache.csv"

# Carregar col·lecció
df = pd.read_csv(CSV_PATH)
df["objectid"] = pd.to_numeric(df["objectid"], errors="coerce").dropna().astype(int)
object_ids = df["objectid"].tolist()
print(f"Total jocs: {len(object_ids)}")

# Carregar caché existent
if os.path.exists(CACHE_PATH):
    try:
        cache_df = pd.read_csv(CACHE_PATH)
        if not cache_df.empty and "objectid" in cache_df.columns:
            cache = dict(zip(
                cache_df["objectid"].astype(int),
                cache_df["mechanics"].apply(eval)
            ))
            print(f"Caché existent: {len(cache)} jocs")
        else:
            cache = {}
    except Exception:
        cache = {}
else:
    cache = {}

ids_to_scrape = [oid for oid in object_ids if oid not in cache]
print(f"Jocs nous a scrapejar: {len(ids_to_scrape)}")

# ============================================================
# MÈTODE 1: BGG JSON API (diferent endpoint, menys bloquejat)
# ============================================================

def fetch_via_json_api(oid: int, session: requests.Session) -> list:
    url = f"https://boardgamegeek.com/boardgame/{oid}"
    # BGG té un endpoint JSON intern que fan servir les seves pròpies pàgines
    api_url = f"https://api.geekdo.com/api/geekitems?objectid={oid}&objecttype=thing&subtype=boardgame"
    try:
        r = session.get(api_url, timeout=15)
        if r.status_code == 200:
            data = r.json()
            links = data.get("item", {}).get("links", {})
            mechanics = [
                m.get("name", "") 
                for m in links.get("boardgamemechanic", [])
            ]
            return [m for m in mechanics if m]
    except Exception as e:
        print(f"      JSON API error: {e}")
    return None

# ============================================================
# MÈTODE 2: Scraping HTML amb headers de navegador real
# ============================================================

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:123.0) Gecko/20100101 Firefox/123.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_3) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Safari/605.1.15",
]

def fetch_via_html(oid: int, session: requests.Session) -> list:
    url = f"https://boardgamegeek.com/boardgame/{oid}"
    headers = {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Referer": "https://boardgamegeek.com/browse/boardgame",
        "DNT": "1",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "same-origin",
    }
    try:
        r = session.get(url, headers=headers, timeout=15)
        if r.status_code == 200:
            soup = BeautifulSoup(r.text, "html.parser")
            mechanics = []
            for a in soup.select("a[href*='/boardgamemechanic/']"):
                text = a.get_text(strip=True)
                if text:
                    mechanics.append(text)
            return list(set(mechanics)) if mechanics else None
    except Exception as e:
        print(f"      HTML scraping error: {e}")
    return None

# ============================================================
# MÈTODE 3: BGG XML API v1 (l'antiga, de vegades menys bloquejada)
# ============================================================

def fetch_via_xml_v1(oid: int, session: requests.Session) -> list:
    import xml.etree.ElementTree as ET
    url = f"https://boardgamegeek.com/xmlapi/boardgame/{oid}?stats=1"
    try:
        r = session.get(url, timeout=15)
        if r.status_code == 200:
            root = ET.fromstring(r.content)
            mechanics = []
            for elem in root.iter("boardgamemechanic"):
                if elem.text:
                    mechanics.append(elem.text.strip())
            return mechanics if mechanics else None
    except Exception as e:
        print(f"      XML v1 error: {e}")
    return None

# ============================================================
# MÈTODE 4: Endpoint de cerca de BGG
# ============================================================

def fetch_via_search_api(oid: int, session: requests.Session) -> list:
    import xml.etree.ElementTree as ET
    url = f"https://boardgamegeek.com/xmlapi2/thing?id={oid}&stats=1"
    headers = {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept": "application/xml,text/xml,*/*",
        "Referer": f"https://boardgamegeek.com/boardgame/{oid}",
    }
    try:
        r = session.get(url, headers=headers, timeout=15)
        if r.status_code == 200:
            root = ET.fromstring(r.content)
            mechanics = []
            for item in root.findall("item"):
                for link in item.findall("link"):
                    if link.get("type") == "boardgamemechanic":
                        val = link.get("value", "")
                        if val:
                            mechanics.append(val)
            return mechanics if mechanics else None
    except Exception as e:
        print(f"      XML v2 error: {e}")
    return None

# ============================================================
# BUCLE PRINCIPAL — prova els 4 mètodes per cada joc
# ============================================================

session = requests.Session()

# Visitar BGG primer per obtenir cookies
print("Obtenint cookies de BGG...")
try:
    session.get("https://boardgamegeek.com", headers={"User-Agent": random.choice(USER_AGENTS)}, timeout=15)
    time.sleep(2)
except Exception:
    pass

methods = [
    ("JSON API",    fetch_via_json_api),
    ("XML v2",      fetch_via_search_api),
    ("XML v1",      fetch_via_xml_v1),
    ("HTML scrape", fetch_via_html),
]

for i, oid in enumerate(ids_to_scrape):
    print(f"\n[{i+1}/{len(ids_to_scrape)}] ID {oid}")
    
    mechanics_found = None
    
    for method_name, method_fn in methods:
        print(f"  Provant {method_name}...")
        result = method_fn(oid, session)
        
        if result is not None:
            mechanics_found = result
            print(f"  ✅ {method_name} — {len(result)} mecàniques: {result[:3]}")
            break
        else:
            print(f"  ❌ {method_name} fallat")
            time.sleep(1)
    
    if mechanics_found is None:
        print(f"  ⚠️ Tots els mètodes han fallat per ID {oid}, guardant llista buida")
        cache[oid] = []
    else:
        cache[oid] = mechanics_found

    # Guardar cada 10 jocs
    if (i + 1) % 10 == 0:
        rows = [{"objectid": k, "mechanics": str(v)} for k, v in cache.items()]
        pd.DataFrame(rows).to_csv(CACHE_PATH, index=False)
        print(f"\n💾 Caché guardada ({len(cache)} jocs)")

    # Pausa aleatòria per semblar més humà
    time.sleep(random.uniform(1.5, 3.5))

# Guardar final
rows = [{"objectid": k, "mechanics": str(v)} for k, v in cache.items()]
pd.DataFrame(rows).to_csv(CACHE_PATH, index=False)

total_amb = sum(1 for v in cache.values() if v)
print(f"\n✅ Fet!")
print(f"  Jocs a la caché: {len(cache)}")
print(f"  Amb mecàniques: {total_amb}")
print(f"  Sense mecàniques: {len(cache) - total_amb}")

Total jocs: 592
Jocs nous a scrapejar: 592
Obtenint cookies de BGG...

[1/592] ID 8203
  Provant JSON API...
  ✅ JSON API — 5 mecàniques: ['Grid Movement', 'Hexagon Grid', 'Map Reduction']

[2/592] ID 146228
  Provant JSON API...
  ✅ JSON API — 3 mecàniques: ['Hexagon Grid', 'Open Drafting', 'Variable Phase Order']

[3/592] ID 340790
  Provant JSON API...
  ✅ JSON API — 3 mecàniques: ['Events', 'Open Drafting', 'Worker Placement']

[4/592] ID 421710
  Provant JSON API...
  ✅ JSON API — 4 mecàniques: ['Action / Event', 'Hand Management', 'Hidden Roles']

[5/592] ID 367280
  Provant JSON API...
  ✅ JSON API — 4 mecàniques: ['Contracts', 'Modular Board', 'Solo / Solitaire Game']

[6/592] ID 85036
  Provant JSON API...
  ✅ JSON API — 4 mecàniques: ['Auction / Bidding', 'Memory', 'Point to Point Movement']

[7/592] ID 371947
  Provant JSON API...
  ✅ JSON API — 5 mecàniques: ['Area Majority / Influence', 'End Game Bonuses', 'Hand Management']

[8/592] ID 327793
  Provant JSON API...
  ✅ JSO